# Lab 1 Exercise: two tools, one loop

This notebook solves the exercise at the end of Lab 1.

Steps:

1. Keep the lab's `get_share_price` tool.
2. Add a fake `get_exchange_rate` tool.
3. Bind both tools to the model.
4. Ask for Amazon's share price in euros.
5. Run the tool loop by hand until the model gives a final answer.

The answer needs a US dollar price and a USD-to-EUR rate, so both tools must run.

## Imports

Use the lab imports. `load_dotenv` finds the `.env` file in the repository root.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv(override=True)

## Model

Use the same model as the lab.

In [ ]:
llm = ChatOpenAI(model="gpt-5.4-mini")

## Share price tool

The lab tool returns a fake price, or `0.0` for an unknown symbol.

In [ ]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price in US dollars for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AMZN"}))

## Exchange rate tool

The new tool returns how many units of a currency equal one US dollar.

In [ ]:
@tool
def get_exchange_rate(currency: str) -> float:
    """Return how many units of the given currency equal 1 US dollar."""
    fake_rates = {"EUR": 0.92, "GBP": 0.79, "JPY": 156.0}
    return fake_rates.get(currency.upper(), 1.0)

print("name:", get_exchange_rate.name)
print("args:", get_exchange_rate.args)
print("called directly:", get_exchange_rate.invoke({"currency": "EUR"}))

## Bind both tools to the model

`bind_tools` makes both tools available to the model. The reply may contain requests in `.tool_calls` instead of a final answer.

In [ ]:
llm_with_tools = llm.bind_tools([get_share_price, get_exchange_rate])

question = "Use both tools to find Amazon's share price in euros."
response = llm_with_tools.invoke(question)
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

## Run the tool loop by hand

The loop:

1. Send the conversation to the model.
2. Add its reply to the conversation.
3. Stop if the reply has no tool calls.
4. Run each tool and add its result as a `ToolMessage`.
5. Repeat.

The name map selects the right tool. The loop allows the model to request tools in one turn or over several turns.

In [ ]:
tools_by_name = {
    "get_share_price": get_share_price,
    "get_exchange_rate": get_exchange_rate,
}

conversation = [HumanMessage(question)]

while True:
    ai_message = llm_with_tools.invoke(conversation)
    conversation.append(ai_message)

    if not ai_message.tool_calls:
        break

    for call in ai_message.tool_calls:
        tool_to_run = tools_by_name[call["name"]]
        result = tool_to_run.invoke(call["args"])
        print(f"Ran {call['name']} with {call['args']} and got {result}")
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

print("\nFinal answer:")
print(conversation[-1].content)